In [1]:
import pandas as pd

# 1. Carregar o arquivo original
df = pd.read_csv('24_CENTRAL_PINHEIRO.csv.csv', sep=None, engine='python')
df = df.dropna(how='all').reset_index(drop=True)

In [2]:
df

,Unidade,Cx.,Ticket,Placa,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Entrada,...,Valor Pago,Pagamento,Tabela,Desconto,Vl. Desconto,Selo,Nr. Selo,Vl. Selo,Vl. Convenio,Rps
0,24 CENTRAL PINHEIRO,1062.0,68540.0,NZN-4544,NaN,NaN,24001.0,CBP BAHIA 2 VAGAS MENSAL 01 SALA 502,NaN,12/28/24 6:54,...,0,NaN,MENSALISTA,NaN,0,NaN,NaN,0,0,0.0
1,24 CENTRAL PINHEIRO,1062.0,68541.0,SJN-3A81,NaN,NaN,240011.0,ANTONIO RAIMUNDO RIBEIRO,NaN,12/30/24 7:34,...,0,NaN,MENSALISTA,NaN,0,NaN,NaN,0,0,0.0
2,24 CENTRAL PINHEIRO,1062.0,68545.0,PLU-9I09,NaN,NaN,240018.0,ADM CENTRAL PINHEIRO,NaN,12/30/24 9:47,...,0,NaN,CONDOMINO,NaN,0,NaN,NaN,0,0,0.0
3,24 CENTRAL PINHEIRO,1062.0,68555.0,SJN-3A81,NaN,NaN,240011.0,ANTONIO RAIMUNDO RIBEIRO,NaN,1/2/25 8:00,...,0,NaN,MENSALISTA,NaN,0,NaN,NaN,0,0,0.0
4,24 CENTRAL PINHEIRO,1063.0,68556.0,SKK-1A06,NaN,NaN,240010.0,AMERICA 70 BLUE BAY REALTY S.A,NaN,1/2/25 8:30,...,0,NaN,MENSALISTA,NaN,0,NaN,NaN,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15860,24 CENTRAL PINHEIRO,1573.0,99189.0,PZB-9G07,NaN,NaN,NaN,AVULSO,NaN,12/30/25 11:53,...,14,CREDITO,ROTATIVO,NaN,0,NaN,NaN,0,0,0.0
15861,24 CENTRAL PINHEIRO,1573.0,99190.0,QTX-7F66,NaN,NaN,NaN,AVULSO,NaN,12/30/25 14:23,...,28,CREDITO,ROTATIVO,NaN,0,NaN,NaN,0,0,0.0
15862,24 CENTRAL PINHEIRO,1573.0,99191.0,TGV-7J70,NaN,NaN,NaN,AVULSO,NaN,12/30/25 15:09,...,14,CREDITO,ROTATIVO,NaN,0,NaN,NaN,0,0,0.0
15863,24 CENTRAL PINHEIRO,1573.0,99192.0,ORC-3J03,NaN,NaN,NaN,AVULSO,NaN,12/30/25 15:45,...,35,CREDITO,ROTATIVO,NaN,0,NaN,NaN,0,0,0.0


In [3]:

# 2. Função de limpeza de moeda (remove R$, pontos e ajusta a vírgula)
def clean_currency(x):
    if pd.isna(x): return 0.0
    if isinstance(x, str):
        clean_val = x.replace('R$', '').replace(' ', '').replace('.', '').replace(',', '.')
        try: return float(clean_val)
        except: return 0.0
    return float(x)

# Aplicar limpeza nas colunas financeiras originais
cols_financeiras = ['Subtotal', 'Valor Pago', 'Vl. Desconto', 'Vl. Selo', 'Vl. Convenio']
for col in cols_financeiras:
    df[col] = df[col].apply(clean_currency)

# 3. Tratamento de Datas (para cálculos internos)
df['Entrada_DT'] = pd.to_datetime(df['Entrada'], errors='coerce')
df['Saida_DT'] = pd.to_datetime(df['Saida'], errors='coerce')
stay_min = (df['Saida_DT'] - df['Entrada_DT']).dt.total_seconds() / 60
stay_hour = (stay_min / 60).apply(lambda x: int(x) + 1 if (x > 0 and x % 1 > 0) else int(x))

# 4. Adicionar colunas de inteligência SEM remover as originais
df['Permanencia_Minutos'] = stay_min.fillna(0)
df['Permanencia_Horas'] = stay_hour.fillna(0)

# 5. Garantir que os nomes das colunas fiquem iguais à imagem
# O pandas já carrega os nomes como estão no CSV. Vamos apenas garantir a ordem.
colunas_finais = [
    'Unidade', 'Cx.', 'Ticket', 'Placa', 'Entrada', 'Saida', 'Permanencia', 
    'Subtotal', 'Valor Pago', 'Pagamento', 'Tabela', 'Vl. Desconto', 
    'Vl. Selo', 'Vl. Convenio', 'Permanencia_Minutos', 'Permanencia_Horas'
]

# Exportar com nomes idênticos à imagem
df[colunas_finais].to_csv('cleaned_parking_data.csv', index=False, sep=';', encoding='latin1')
print("Arquivo para Power BI gerado com colunas idênticas!")

C:\Users\PC\AppData\Local\Temp\ipykernel_18396\1578863983.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Entrada_DT'] = pd.to_datetime(df['Entrada'], errors='coerce')
C:\Users\PC\AppData\Local\Temp\ipykernel_18396\1578863983.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Saida_DT'] = pd.to_datetime(df['Saida'], errors='coerce')


ValueError: cannot convert float NaN to integer

In [4]:
import pandas as pd
import numpy as np

# 1. Carregar o arquivo original
df = pd.read_csv('24_CENTRAL_PINHEIRO.csv.csv', sep=None, engine='python')

# Remover linhas totalmente vazias (comum em exportações de sistemas)
df = df.dropna(subset=['Entrada', 'Saida', 'Ticket'], how='all').reset_index(drop=True)

# 2. Limpeza de Moeda
def clean_currency(x):
    if pd.isna(x) or x == '': return 0.0
    if isinstance(x, str):
        # Remove R$, espaços, pontos de milhar e troca vírgula por ponto
        clean_val = x.replace('R$', '').replace(' ', '').replace('.', '').replace(',', '.')
        try: return float(clean_val)
        except: return 0.0
    return float(x)

cols_financeiras = ['Subtotal', 'Valor Pago', 'Vl. Desconto', 'Vl. Selo', 'Vl. Convenio']
for col in cols_financeiras:
    df[col] = df[col].apply(clean_currency)

# 3. Tratamento de Datas (Corrigindo o Warning de formato)
# dayfirst=False pois o padrão do seu arquivo parece ser MM/DD/YY (ex: 12/28/24)
df['Entrada_DT'] = pd.to_datetime(df['Entrada'], errors='coerce')
df['Saida_DT'] = pd.to_datetime(df['Saida'], errors='coerce')

# 4. Cálculo de Permanência com tratamento para NaN
# Calculamos os minutos
df['Permanencia_Minutos'] = (df['Saida_DT'] - df['Entrada_DT']).dt.total_seconds() / 60
df['Permanencia_Minutos'] = df['Permanencia_Minutos'].fillna(0)

# Cálculo de Horas arredondado para cima (Corrigindo o ValueError)
def calcular_hora_cheia(minutos):
    if minutos <= 0:
        return 0
    # math.ceil manual para evitar erros de tipo
    horas = minutos / 60
    if horas % 1 > 0:
        return int(horas) + 1
    return int(horas)

df['Permanencia_Horas'] = df['Permanencia_Minutos'].apply(calcular_hora_cheia)

# 5. Organização Final das Colunas (Igual à sua imagem)
colunas_finais = [
    'Unidade', 'Cx.', 'Ticket', 'Placa', 'Entrada', 'Saida', 'Permanencia', 
    'Subtotal', 'Valor Pago', 'Pagamento', 'Tabela', 'Vl. Desconto', 
    'Vl. Selo', 'Vl. Convenio', 'Permanencia_Minutos', 'Permanencia_Horas'
]

# Garantir que todas as colunas existem antes de salvar
colunas_existentes = [c for c in colunas_finais if c in df.columns]

# Exportar
df[colunas_existentes].to_csv('cleaned_parking_data.csv', index=False, sep=';', encoding='latin1')

print("✅ Sucesso! Arquivo 'cleaned_parking_data.csv' gerado sem erros.")

C:\Users\PC\AppData\Local\Temp\ipykernel_18396\452691627.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Entrada_DT'] = pd.to_datetime(df['Entrada'], errors='coerce')
C:\Users\PC\AppData\Local\Temp\ipykernel_18396\452691627.py:27: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Saida_DT'] = pd.to_datetime(df['Saida'], errors='coerce')


✅ Sucesso! Arquivo 'cleaned_parking_data.csv' gerado sem erros.


In [5]:
df

,Unidade,Cx.,Ticket,Placa,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Entrada,...,Vl. Desconto,Selo,Nr. Selo,Vl. Selo,Vl. Convenio,Rps,Entrada_DT,Saida_DT,Permanencia_Minutos,Permanencia_Horas
0,24 CENTRAL PINHEIRO,1062.0,68540.0,NZN-4544,NaN,NaN,24001.0,CBP BAHIA 2 VAGAS MENSAL 01 SALA 502,NaN,12/28/24 6:54,...,0.0,NaN,NaN,0.0,0.0,0.0,2024-12-28 06:54:00,2025-01-02 09:08:00,7334.0,123
1,24 CENTRAL PINHEIRO,1062.0,68541.0,SJN-3A81,NaN,NaN,240011.0,ANTONIO RAIMUNDO RIBEIRO,NaN,12/30/24 7:34,...,0.0,NaN,NaN,0.0,0.0,0.0,2024-12-30 07:34:00,2025-01-02 07:04:00,4290.0,72
2,24 CENTRAL PINHEIRO,1062.0,68545.0,PLU-9I09,NaN,NaN,240018.0,ADM CENTRAL PINHEIRO,NaN,12/30/24 9:47,...,0.0,NaN,NaN,0.0,0.0,0.0,2024-12-30 09:47:00,2025-01-02 06:42:00,4135.0,69
3,24 CENTRAL PINHEIRO,1062.0,68555.0,SJN-3A81,NaN,NaN,240011.0,ANTONIO RAIMUNDO RIBEIRO,NaN,1/2/25 8:00,...,0.0,NaN,NaN,0.0,0.0,0.0,2025-01-02 08:00:00,2025-01-02 13:33:00,333.0,6
4,24 CENTRAL PINHEIRO,1063.0,68556.0,SKK-1A06,NaN,NaN,240010.0,AMERICA 70 BLUE BAY REALTY S.A,NaN,1/2/25 8:30,...,0.0,NaN,NaN,0.0,0.0,0.0,2025-01-02 08:30:00,2025-01-02 18:51:00,621.0,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15859,24 CENTRAL PINHEIRO,1573.0,99188.0,PKT-2236,NaN,NaN,NaN,AVULSO,NaN,12/30/25 11:33,...,0.0,NaN,NaN,0.0,0.0,0.0,2025-12-30 11:33:00,2025-12-30 13:16:00,103.0,2
15860,24 CENTRAL PINHEIRO,1573.0,99189.0,PZB-9G07,NaN,NaN,NaN,AVULSO,NaN,12/30/25 11:53,...,0.0,NaN,NaN,0.0,0.0,0.0,2025-12-30 11:53:00,2025-12-30 12:44:00,51.0,1
15861,24 CENTRAL PINHEIRO,1573.0,99190.0,QTX-7F66,NaN,NaN,NaN,AVULSO,NaN,12/30/25 14:23,...,0.0,NaN,NaN,0.0,0.0,0.0,2025-12-30 14:23:00,2025-12-30 16:02:00,99.0,2
15862,24 CENTRAL PINHEIRO,1573.0,99191.0,TGV-7J70,NaN,NaN,NaN,AVULSO,NaN,12/30/25 15:09,...,0.0,NaN,NaN,0.0,0.0,0.0,2025-12-30 15:09:00,2025-12-30 16:01:00,52.0,1


In [6]:
df[colunas_existentes]

,Unidade,Cx.,Ticket,Placa,Entrada,Saida,Permanencia,Subtotal,Valor Pago,Pagamento,Tabela,Vl. Desconto,Vl. Selo,Vl. Convenio,Permanencia_Minutos,Permanencia_Horas
0,24 CENTRAL PINHEIRO,1062.0,68540.0,NZN-4544,12/28/24 6:54,1/2/25 9:08,122:14:16,0.0,0.0,NaN,MENSALISTA,0.0,0.0,0.0,7334.0,123
1,24 CENTRAL PINHEIRO,1062.0,68541.0,SJN-3A81,12/30/24 7:34,1/2/25 7:04,71:29:47,0.0,0.0,NaN,MENSALISTA,0.0,0.0,0.0,4290.0,72
2,24 CENTRAL PINHEIRO,1062.0,68545.0,PLU-9I09,12/30/24 9:47,1/2/25 6:42,68:55:30,0.0,0.0,NaN,CONDOMINO,0.0,0.0,0.0,4135.0,69
3,24 CENTRAL PINHEIRO,1062.0,68555.0,SJN-3A81,1/2/25 8:00,1/2/25 13:33,5:33:02,0.0,0.0,NaN,MENSALISTA,0.0,0.0,0.0,333.0,6
4,24 CENTRAL PINHEIRO,1063.0,68556.0,SKK-1A06,1/2/25 8:30,1/2/25 18:51,10:20:56,0.0,0.0,NaN,MENSALISTA,0.0,0.0,0.0,621.0,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15859,24 CENTRAL PINHEIRO,1573.0,99188.0,PKT-2236,12/30/25 11:33,12/30/25 13:16,1:43:23,0.0,28.0,DEBITO,ROTATIVO,0.0,0.0,0.0,103.0,2
15860,24 CENTRAL PINHEIRO,1573.0,99189.0,PZB-9G07,12/30/25 11:53,12/30/25 12:44,0:51:27,0.0,14.0,CREDITO,ROTATIVO,0.0,0.0,0.0,51.0,1
15861,24 CENTRAL PINHEIRO,1573.0,99190.0,QTX-7F66,12/30/25 14:23,12/30/25 16:02,1:38:13,0.0,28.0,CREDITO,ROTATIVO,0.0,0.0,0.0,99.0,2
15862,24 CENTRAL PINHEIRO,1573.0,99191.0,TGV-7J70,12/30/25 15:09,12/30/25 16:01,0:52:06,0.0,14.0,CREDITO,ROTATIVO,0.0,0.0,0.0,52.0,1


In [7]:
# Tabela de referência (Mediana de Salvador para este perfil de cliente)
precos_referencia = {1: 12, 2: 21, 3: 35, 4: 48, 5: 54, 6: 66}

def verificar_vazamento(row):
    # 'Tabela' na imagem é onde diz se é ROTATIVO ou MENSALISTA
    if row['Tabela'] != 'ROTATIVO' or row['Permanencia_Minutos'] <= 15: 
        return False
    
    h = row['Permanencia_Horas']
    pago = row['Valor Pago']
    
    # Se o valor pago for menor que a tabela esperada
    if h == 1 and pago < 10: return True
    if h == 2 and pago < 18: return True
    if h >= 3 and pago < 25: return True
    return False

def calcular_perda(row):
    if not row['Possivel_Vazamento']: return 0
    h = row['Permanencia_Horas']
    esperado = precos_referencia.get(h, 70 if h > 0 else 0)
    return max(0, esperado - row['Valor Pago'])

# Aplicar lógica nas colunas com nomes da imagem
df['Possivel_Vazamento'] = df.apply(verificar_vazamento, axis=1)
df['Perda_Estimada_R$'] = df.apply(calcular_perda, axis=1)

# Exportar
df.to_csv('relatorio_final_analise.csv', index=False, sep=';', encoding='latin1')
print("Relatório de Auditoria com nomes de colunas originais gerado!")

Relatório de Auditoria com nomes de colunas originais gerado!


In [8]:
import pandas as pd

# 1. Converte as colunas para datetime
# O format='mixed' ajuda se houver formatos diferentes na mesma coluna (comum em versões recentes do pandas)
df['Entrada'] = pd.to_datetime(df['Entrada'], dayfirst=False, errors='coerce')
df['Saida'] = pd.to_datetime(df['Saida'], dayfirst=False, errors='coerce')

# 2. Agora você pode exportar para Excel
df.to_excel('seu_arquivo_padronizado.xlsx', index=False)

C:\Users\PC\AppData\Local\Temp\ipykernel_18396\1840850306.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Entrada'] = pd.to_datetime(df['Entrada'], dayfirst=False, errors='coerce')
C:\Users\PC\AppData\Local\Temp\ipykernel_18396\1840850306.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Saida'] = pd.to_datetime(df['Saida'], dayfirst=False, errors='coerce')


In [9]:
df

,Unidade,Cx.,Ticket,Placa,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Entrada,...,Nr. Selo,Vl. Selo,Vl. Convenio,Rps,Entrada_DT,Saida_DT,Permanencia_Minutos,Permanencia_Horas,Possivel_Vazamento,Perda_Estimada_R$
0,24 CENTRAL PINHEIRO,1062.0,68540.0,NZN-4544,NaN,NaN,24001.0,CBP BAHIA 2 VAGAS MENSAL 01 SALA 502,NaN,2024-12-28 06:54:00,...,NaN,0.0,0.0,0.0,2024-12-28 06:54:00,2025-01-02 09:08:00,7334.0,123,False,0.0
1,24 CENTRAL PINHEIRO,1062.0,68541.0,SJN-3A81,NaN,NaN,240011.0,ANTONIO RAIMUNDO RIBEIRO,NaN,2024-12-30 07:34:00,...,NaN,0.0,0.0,0.0,2024-12-30 07:34:00,2025-01-02 07:04:00,4290.0,72,False,0.0
2,24 CENTRAL PINHEIRO,1062.0,68545.0,PLU-9I09,NaN,NaN,240018.0,ADM CENTRAL PINHEIRO,NaN,2024-12-30 09:47:00,...,NaN,0.0,0.0,0.0,2024-12-30 09:47:00,2025-01-02 06:42:00,4135.0,69,False,0.0
3,24 CENTRAL PINHEIRO,1062.0,68555.0,SJN-3A81,NaN,NaN,240011.0,ANTONIO RAIMUNDO RIBEIRO,NaN,2025-01-02 08:00:00,...,NaN,0.0,0.0,0.0,2025-01-02 08:00:00,2025-01-02 13:33:00,333.0,6,False,0.0
4,24 CENTRAL PINHEIRO,1063.0,68556.0,SKK-1A06,NaN,NaN,240010.0,AMERICA 70 BLUE BAY REALTY S.A,NaN,2025-01-02 08:30:00,...,NaN,0.0,0.0,0.0,2025-01-02 08:30:00,2025-01-02 18:51:00,621.0,11,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15859,24 CENTRAL PINHEIRO,1573.0,99188.0,PKT-2236,NaN,NaN,NaN,AVULSO,NaN,2025-12-30 11:33:00,...,NaN,0.0,0.0,0.0,2025-12-30 11:33:00,2025-12-30 13:16:00,103.0,2,False,0.0
15860,24 CENTRAL PINHEIRO,1573.0,99189.0,PZB-9G07,NaN,NaN,NaN,AVULSO,NaN,2025-12-30 11:53:00,...,NaN,0.0,0.0,0.0,2025-12-30 11:53:00,2025-12-30 12:44:00,51.0,1,False,0.0
15861,24 CENTRAL PINHEIRO,1573.0,99190.0,QTX-7F66,NaN,NaN,NaN,AVULSO,NaN,2025-12-30 14:23:00,...,NaN,0.0,0.0,0.0,2025-12-30 14:23:00,2025-12-30 16:02:00,99.0,2,False,0.0
15862,24 CENTRAL PINHEIRO,1573.0,99191.0,TGV-7J70,NaN,NaN,NaN,AVULSO,NaN,2025-12-30 15:09:00,...,NaN,0.0,0.0,0.0,2025-12-30 15:09:00,2025-12-30 16:01:00,52.0,1,False,0.0


In [1]:
df['Pagamento'] = df['Pagamento'].fillna('sem_pagto')

NameError: name 'df' is not defined